In [10]:
# Cell 1 — imports, config, and robust model loader
import os, joblib, json, re
import numpy as np, pandas as pd
from datetime import datetime
from pathlib import Path

# ---- Config: candidate paths (edit if your files have other names) ----
CANDIDATE_PREPROCESSORS = [
    "models/preprocessor.joblib",
    "models/preprocessor_sample.joblib",
    "models/preprocessor_full.joblib",
    "models/preprocessor_sample_v1.joblib"
]
CANDIDATE_CLASSIFIERS = [
    "models/rf_classifier.joblib",
    "models/rf_classifier_sample.joblib",
    "models/rf_classifier_full.joblib"
]
CANDIDATE_REGRESSORS = [
    "models/rf_regressor.joblib",
    "models/rf_regressor_sample.joblib",
    "models/rf_regressor_full.joblib"
]

# Optional helpers (may not exist)
TOP_ORIG_CANDIDATES = ["models/top_orig.joblib", "models/top_orig_sample.joblib"]
TOP_DEST_CANDIDATES = ["models/top_dest.joblib", "models/top_dest_sample.joblib"]
IATA_TO_ID_CANDIDATES = ["models/iata_to_id.joblib"]

# Function to find first existing path from a list
def find_existing(paths):
    for p in paths:
        if Path(p).exists():
            return p
    return None

# Load preprocessor, clf, regressor (fail fast with helpful message)
preproc_path = find_existing(CANDIDATE_PREPROCESSORS)
clf_path = find_existing(CANDIDATE_CLASSIFIERS)
reg_path = find_existing(CANDIDATE_REGRESSORS)

if not preproc_path or not clf_path or not reg_path:
    print("Missing model artifact(s). Searched for:")
    print(" Preprocessors:", CANDIDATE_PREPROCESSORS)
    print(" Classifiers:  ", CANDIDATE_CLASSIFIERS)
    print(" Regressors:   ", CANDIDATE_REGRESSORS)
    raise FileNotFoundError("One or more required model artifacts not found. Place them under models/ or update paths in Cell 1.")

preprocessor = joblib.load(preproc_path)
clf = joblib.load(clf_path)
reg = joblib.load(reg_path)
print("Loaded preprocessor from:", preproc_path)
print("Loaded clf from:", clf_path)
print("Loaded reg from:", reg_path)

# Attempt to load optional top-K lists and iata->id map (if exist)
top_orig_path = find_existing(TOP_ORIG_CANDIDATES)
top_dest_path = find_existing(TOP_DEST_CANDIDATES)
iata_map_path = find_existing(IATA_TO_ID_CANDIDATES)

top_orig = joblib.load(top_orig_path) if top_orig_path else []
top_dest = joblib.load(top_dest_path) if top_dest_path else []
iata_to_id = joblib.load(iata_map_path) if iata_map_path else {}

print("top_orig loaded:", bool(top_orig), "top_dest loaded:", bool(top_dest), "iata_to_id loaded:", bool(iata_to_id))

# Defaults for numeric fields (if you saved processed data medians, you can set them here)
DEFAULT_DISTANCE = 800      # km-ish placeholder
DEFAULT_CRS_ELAPSED = 120  # minutes
# If you have a processed sample file, try to extract medians automatically (non-fatal)
possible_processed = ["data/processed_flights_sample.parquet", "data/processed_flights.parquet"]
for p in possible_processed:
    if Path(p).exists():
        try:
            tmp = pd.read_parquet(p)
            if 'DISTANCE' in tmp.columns:
                DEFAULT_DISTANCE = float(tmp['DISTANCE'].median())
            if 'CRS_ELAPSED_TIME' in tmp.columns:
                DEFAULT_CRS_ELAPSED = float(tmp['CRS_ELAPSED_TIME'].median())
            print("Loaded medians from", p)
            break
        except Exception:
            pass

print("Defaults: DISTANCE=", DEFAULT_DISTANCE, "CRS_ELAPSED_TIME=", DEFAULT_CRS_ELAPSED)

Loaded preprocessor from: models/preprocessor_sample.joblib
Loaded clf from: models/rf_classifier_sample.joblib
Loaded reg from: models/rf_regressor_sample.joblib
top_orig loaded: False top_dest loaded: False iata_to_id loaded: False
Loaded medians from data/processed_flights_sample.parquet
Defaults: DISTANCE= 800 CRS_ELAPSED_TIME= 120


In [11]:
# Cell 2 — rule-based parser (no external APIs)
import re

def parse_query_rule_based(text):
    """
    Returns dict: carrier, origin (IATA), destination (IATA),
    departure_hour (0-23) or None, dayofweek (0=Mon..6=Sun) or None
    """
    out = {'carrier': None, 'origin': None, 'destination': None, 'departure_hour': None, 'dayofweek': None}
    if not text or not isinstance(text, str):
        return out
    t_upper = text.upper()

    # carrier: prefer two-letter code tokens (AA, DL, UA...) — naive but works for demos
    m = re.search(r"\b([A-Z]{2})\b", t_upper)
    if m:
        out['carrier'] = m.group(1)

    # IATA codes (3-letter) detection
    iatas = re.findall(r"\b([A-Z]{3})\b", t_upper)
    m_from = re.search(r"FROM\s+([A-Z]{3})", t_upper)
    m_to   = re.search(r"TO\s+([A-Z]{3})", t_upper)
    if m_from:
        out['origin'] = m_from.group(1)
    elif iatas:
        out['origin'] = iatas[0]
    if m_to:
        out['destination'] = m_to.group(1)
    elif len(iatas) > 1:
        out['destination'] = iatas[1]

    # times: morning/afternoon heuristics or explicit times
    if re.search(r"\bmorning\b", t_upper):
        out['departure_hour'] = 9
    elif re.search(r"\bafternoon\b", t_upper):
        out['departure_hour'] = 15
    else:
        m_time = re.search(r"(\d{1,2})(?::|\.?)(\d{2})?\s*(AM|PM)?", text, flags=re.I)
        if m_time:
            hh = int(m_time.group(1))
            ampm = (m_time.group(3) or "").upper()
            if ampm == "PM" and hh < 12:
                hh += 12
            if ampm == "AM" and hh == 12:
                hh = 0
            out['departure_hour'] = hh

    # day of week heuristic
    days = {"MON":0,"TUE":1,"WED":2,"THU":3,"FRI":4,"SAT":5,"SUN":6}
    for k,v in days.items():
        if k in t_upper:
            out['dayofweek'] = v
            break

    return out

# quick sanity check
print(parse_query_rule_based("Will my AA flight from JFK to LAX tomorrow morning be delayed?"))

{'carrier': 'MY', 'origin': 'JFK', 'destination': 'LAX', 'departure_hour': None, 'dayofweek': None}


In [15]:
# Cell 3 — mapping helpers and building the one-row DataFrame for model input
import pandas as pd

def iata_to_topk_id(iata, top_list):
    """
    Map IATA -> numeric airport ID in top_list if mapping exists, else return -1 (OTHER).
    Requires iata_to_id mapping to be present for useful mapping.
    """
    if not iata:
        return -1
    iata_u = iata.upper()
    if iata_u in iata_to_id:
        iid = iata_to_id[iata_u]
        if iid in top_list:
            return iid
        return -1
    return -1

def build_model_row(parsed):
    """
    Returns a pandas DataFrame with a single row matching preprocessor expected features.
    Uses defaults for numeric fields if medians are not available.
    """
    now = datetime.now()
    row = {}
    row['YEAR'] = now.year
    row['month'] = now.month
    row['dayofweek'] = parsed.get('dayofweek', now.weekday())
    row['sched_dep_hour'] = parsed.get('departure_hour', 12)
    # numeric defaults
    row['DISTANCE'] = DEFAULT_DISTANCE
    row['CRS_ELAPSED_TIME'] = DEFAULT_CRS_ELAPSED
    # origin/dest: prefer top lists (may be empty -> will be -1)
    origin_top = iata_to_topk_id(parsed.get('origin'), top_orig)
    dest_top   = iata_to_topk_id(parsed.get('destination'), top_dest)
    # If top lists are empty but user provided numeric ID directly, accept numeric values
    # allow user to pass numeric origin in parsed['origin'] (if they prefer)
    try:
        if origin_top == -1 and parsed.get('origin') and str(parsed.get('origin')).isdigit():
            origin_top = int(parsed.get('origin'))
    except Exception:
        pass
    try:
        if dest_top == -1 and parsed.get('destination') and str(parsed.get('destination')).isdigit():
            dest_top = int(parsed.get('destination'))
    except Exception:
        pass

    row['origin_topK'] = origin_top
    row['dest_topK'] = dest_top
    row['OP_UNIQUE_CARRIER'] = parsed.get('carrier') or 'OTHER'
    return pd.DataFrame([row])

# quick test
parsed = parse_query_rule_based("Will AA flight from JFK to LAX tomorrow morning be delayed?")
print(parsed)
print(build_model_row(parsed))

{'carrier': 'AA', 'origin': 'JFK', 'destination': 'LAX', 'departure_hour': None, 'dayofweek': None}
   YEAR  month dayofweek sched_dep_hour  DISTANCE  CRS_ELAPSED_TIME  \
0  2026      1      None           None       800               120   

   origin_topK  dest_topK OP_UNIQUE_CARRIER  
0           -1         -1                AA  


In [17]:
# Cell 4 — run preprocessor + models and produce human-friendly output
CLASS_MAP = {0: "On-time (≤ 15 min)", 1: "Minor delay (16–59 min)", 2: "Major delay (60+ min)"}

def predict_from_row(df_row):
    Xp = preprocessor.transform(df_row)
    cls = int(clf.predict(Xp)[0])
    minutes = float(reg.predict(Xp)[0])
    prob = None
    try:
        prob = float(clf.predict_proba(Xp)[0].max())
    except Exception:
        prob = None
    return {"class": cls, "minutes": minutes, "prob": prob}

def answer_query_text(text):
    parsed = parse_query_rule_based(text)
    model_row = build_model_row(parsed)
    pred = predict_from_row(model_row)
    s = (
        f"Parsed: {parsed}\n"
        f"Prediction: {CLASS_MAP[pred['class']]}\n"
        f"Estimated delay: {pred['minutes']:.1f} minutes\n"
        f"Confidence (clf max prob): {pred['prob']}"
    )
    return s

# quick demo
print(answer_query_text("Will AA flight from JFK to LAX tomorrow morning be delayed?"))

Parsed: {'carrier': 'AA', 'origin': 'JFK', 'destination': 'LAX', 'departure_hour': None, 'dayofweek': None}
Prediction: Minor delay (16–59 min)
Estimated delay: 6.4 minutes
Confidence (clf max prob): 0.3626816720112966


/opt/homebrew/Cellar/jupyterlab/4.3.1_1/libexec/lib/python3.12/site-packages/sklearn/impute/_base.py:653: UserWarning: Skipping features without any observed values: ['month' 'dayofweek' 'sched_dep_hour']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


In [14]:
# Cell 5 — demo queries
queries = [
    "Will DL flight from JFK to LAX tomorrow morning be delayed?",
    "Is my AA flight from ORD to SFO at 7am likely to be delayed?",
    "Flight from ATL to MCO this evening — delay?",
    "WN flight from DEN to LAS tomorrow afternoon"
]
for q in queries:
    print("Q:", q)
    print(answer_query_text(q))
    print("-" * 50)

Q: Will DL flight from JFK to LAX tomorrow morning be delayed?
Parsed: {'carrier': 'DL', 'origin': 'JFK', 'destination': 'LAX', 'departure_hour': None, 'dayofweek': None}
Prediction: Minor delay (16–59 min)
Estimated delay: 6.4 minutes
Confidence (clf max prob): 0.3626816720112966
--------------------------------------------------
Q: Is my AA flight from ORD to SFO at 7am likely to be delayed?
Parsed: {'carrier': 'IS', 'origin': 'ORD', 'destination': 'SFO', 'departure_hour': 7, 'dayofweek': None}
Prediction: Minor delay (16–59 min)
Estimated delay: 6.4 minutes
Confidence (clf max prob): 0.3626816720112966
--------------------------------------------------
Q: Flight from ATL to MCO this evening — delay?
Parsed: {'carrier': 'TO', 'origin': 'ATL', 'destination': 'MCO', 'departure_hour': None, 'dayofweek': None}
Prediction: Minor delay (16–59 min)
Estimated delay: 6.4 minutes
Confidence (clf max prob): 0.3626816720112966
--------------------------------------------------
Q: WN flight from 

/opt/homebrew/Cellar/jupyterlab/4.3.1_1/libexec/lib/python3.12/site-packages/sklearn/impute/_base.py:653: UserWarning: Skipping features without any observed values: ['month' 'dayofweek' 'sched_dep_hour']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/opt/homebrew/Cellar/jupyterlab/4.3.1_1/libexec/lib/python3.12/site-packages/sklearn/impute/_base.py:653: UserWarning: Skipping features without any observed values: ['month' 'dayofweek' 'sched_dep_hour']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/opt/homebrew/Cellar/jupyterlab/4.3.1_1/libexec/lib/python3.12/site-packages/sklearn/impute/_base.py:653: UserWarning: Skipping features without any observed values: ['month' 'dayofweek' 'sched_dep_hour']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/opt/homebrew/Cellar/jupyterlab/4.3.1_1/libexec/lib/python3.12/site-packages/sklearn/impute